# PRISM v2.1: Prospective-Retrospective Introspective Self-Model

**Kaggle Community Benchmark — Metacognition Track**
*Measuring Progress Toward AGI: Cognitive Abilities Hackathon*

This notebook implements a metacognition benchmark that measures whether LLMs can accurately monitor their own reasoning — before, during, and after solving multi-step mathematical problems.

It operationalizes the Nelson & Narens (1990) monitoring/control framework across six complementary tasks:

| Task | What it measures | Metric | Range |
|------|-----------------|--------|-------|
| T1 | Prospective Calibration | AUROC | 0-1 (chance=0.5) |
| T2 | Step-Level Prospective Accuracy | Spearman rho | 0-1 (chance=0.5) |
| T3 | Retrospective Self-Assessment | Proportion correct | 0-1 |
| T4 | **Prospective-Retrospective Coherence** | Weighted composite | 0-1 |
| T5 | Adaptive Calibration | Pearson r | 0-1 |
| T6 | Novelty Robustness | L2/L1 ratio | 0-1 |

**Task 4 (Coherence)** is the primary leaderboard metric — it is the most novel score, measuring internal consistency between before-task and after-task metacognitive reports.

In [ ]:
import json
import os
import sys

import kaggle_benchmarks as kbench

# Add the prism_v2 package to path if running from dataset
# On Kaggle, the dataset is mounted at /kaggle/input/<dataset-name>/
PRISM_PATHS = [
    "/kaggle/input/prism-v2-benchmark",
    "/kaggle/input/prism-v2",
    ".",
]
for path in PRISM_PATHS:
    if os.path.isdir(os.path.join(path, "prism_v2")):
        sys.path.insert(0, path)
        break

from prism_v2.problems.generator import generate_all_problems
from prism_v2.pipeline import PrismPipeline
from prism_v2.tasks.task_01_prospective_calibration import compute_task_1
from prism_v2.tasks.task_02_step_accuracy import compute_task_2
from prism_v2.tasks.task_03_retrospective_accuracy import compute_task_3
from prism_v2.tasks.task_04_coherence import compute_task_4
from prism_v2.tasks.task_05_adaptive_calibration import compute_task_5
from prism_v2.tasks.task_06_novelty_robustness import compute_task_6

## Load Problem Sets

In [ ]:
def load_problems():
    """Load problem sets from JSON files or generate them."""
    problem_paths = [
        "/kaggle/input/prism-v2-benchmark/prism_v2/problems",
        "/kaggle/input/prism-v2/prism_v2/problems",
        "prism_v2/problems",
    ]

    for base_path in problem_paths:
        l1_path = os.path.join(base_path, "l1_problems.json")
        l2_path = os.path.join(base_path, "l2_problems.json")
        if os.path.exists(l1_path) and os.path.exists(l2_path):
            with open(l1_path) as f:
                l1 = json.load(f)
            with open(l2_path) as f:
                l2 = json.load(f)
            print(f"Loaded problems from {base_path}")
            return l1, l2

    # Fall back to generation
    print("Generating problems at runtime...")
    data = generate_all_problems(base_seed=42, num_main=10, num_feedback=5)
    return data["l1"], data["l2"]


l1_problems, l2_problems = load_problems()
print(
    f"L1 problems: {len(l1_problems)} "
    f"(main: {sum(1 for p in l1_problems if 'feedback' not in p['id'])}, "
    f"feedback: {sum(1 for p in l1_problems if 'feedback' in p['id'])})"
)
print(
    f"L2 problems: {len(l2_problems)} "
    f"(main: {sum(1 for p in l2_problems if 'feedback' not in p['id'])}, "
    f"feedback: {sum(1 for p in l2_problems if 'feedback' in p['id'])})"
)

## Initialize Pipeline

In [ ]:
pipeline = PrismPipeline(l1_problems, l2_problems)

## Define the Benchmark Task

The main task runs the full D1 (Prospective) -> D2 (Performance) -> D3 (Retrospective) pipeline for all 30 problems across both novelty levels, then computes and reports all six metric scores.

The returned value is the **Task 4 (Coherence) composite score**, which serves as the primary leaderboard metric.

In [ ]:
@kbench.task(name="prism_metacognition")
def prism_metacognition(llm) -> float:
    """
    PRISM v2.1: Prospective-Retrospective Introspective Self-Model

    A metacognition benchmark measuring whether LLMs can accurately
    monitor their own reasoning before, during, and after solving problems.

    Returns the Task 4 (Coherence) score as the primary metric,
    while computing and reporting all six task scores.
    """
    # Run the full pipeline (D1 -> D2 -> D3 for all problems)
    # Uses caching to avoid redundant LLM calls on re-runs
    with kbench.client.enable_cache():
        pipeline.run_all(llm, kbench)

    # --- Compute all six task scores ---

    # L1 (familiar) scores
    t1_l1 = compute_task_1(pipeline, novelty_level=1)
    t2_l1 = compute_task_2(pipeline, novelty_level=1)
    t3_l1 = compute_task_3(pipeline, novelty_level=1)
    t4_l1 = compute_task_4(pipeline, kbench, judge_llm=kbench.judge_llm, novelty_level=1)
    t5_l1 = compute_task_5(pipeline, novelty_level=1)

    # L2 (novel operator) scores
    t1_l2 = compute_task_1(pipeline, novelty_level=2)
    t2_l2 = compute_task_2(pipeline, novelty_level=2)
    t3_l2 = compute_task_3(pipeline, novelty_level=2)
    t4_l2 = compute_task_4(pipeline, kbench, judge_llm=kbench.judge_llm, novelty_level=2)
    t5_l2 = compute_task_5(pipeline, novelty_level=2)

    # Cross-level score
    t6 = compute_task_6(pipeline, kbench, judge_llm=kbench.judge_llm)

    # --- Report all scores via assertions for leaderboard visibility ---
    kbench.assertions.assert_true(
        True,
        expectation=f"T1 Prospective Calibration (AUROC): L1={t1_l1:.3f}, L2={t1_l2:.3f}",
    )
    kbench.assertions.assert_true(
        True,
        expectation=f"T2 Step-Level Accuracy (Spearman): L1={t2_l1:.3f}, L2={t2_l2:.3f}",
    )
    kbench.assertions.assert_true(
        True,
        expectation=f"T3 Retrospective Accuracy: L1={t3_l1:.3f}, L2={t3_l2:.3f}",
    )
    kbench.assertions.assert_true(
        True,
        expectation=f"T4 Coherence (composite): L1={t4_l1:.3f}, L2={t4_l2:.3f}",
    )
    kbench.assertions.assert_true(
        True,
        expectation=f"T5 Adaptive Calibration: L1={t5_l1:.3f}, L2={t5_l2:.3f}",
    )
    kbench.assertions.assert_true(
        True,
        expectation=f"T6 Novelty Robustness: {t6:.3f}",
    )

    # Report pipeline summary
    summary = pipeline.summary()
    kbench.assertions.assert_true(
        True,
        expectation=(
            f"Pipeline: {summary['l1_main_count']} L1 + {summary['l2_main_count']} L2 problems | "
            f"Accuracy L1={summary['l1_overall_accuracy']:.1%} L2={summary['l2_overall_accuracy']:.1%} | "
            f"Parse errors={summary['parse_error_rate']:.1%}"
        ),
    )

    # Return Task 4 (Coherence) as the primary score
    return t4_l1

## Run the Benchmark

In [ ]:
run = prism_metacognition.run(llm=kbench.llm)
run

## Select Task for Leaderboard

In [ ]:
%choose prism_metacognition